In [ ]:
# Cell 1 — Installation corrigée
!pip install unsloth
!pip install --upgrade --force-reinstall pillow==11.3.0  # forcer la version compatible

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import json
import re
import random
from datasets import Dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig

In [ ]:
# ============================================================
# CELL 3 — Préparation des données
# ============================================================

import os
from pathlib import Path

def clean_text(text):
    text = re.sub(r"\\n\\n", "\n\n", text)
    text = re.sub(r"\\n", "\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_article_content(text):
    match = re.search(r"Article\s+\S+\s*-\s*(.+)", text, re.DOTALL | re.IGNORECASE)
    if match:
        return clean_text(match.group(1))
    return clean_text(text)

def generate_examples(entry):
    titre       = entry.get("text_juridique_name", "").strip()
    domaine     = entry.get("legal_field", "").strip()
    article_num = entry.get("article_num", "").strip()
    chapter     = entry.get("chapter", "").strip()
    status      = entry.get("status", "").strip().lower()
    contenu     = extract_article_content(entry.get("text", ""))

    if not contenu or not titre:
        return []

    system = (
        "Tu es un assistant juridique spécialisé en droit ivoirien. "
        "Tu connais les lois, décrets et règlements de la République de Côte d'Ivoire. "
        "Réponds avec précision en citant les textes applicables."
    )

    examples = [
        {"conversations": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": f"Que dit l'article {article_num} de la {titre} ?"},
            {"role": "assistant", "content": f"L'article {article_num} de la {titre} ({status}) prévoit :\n\n{contenu}"}
        ]},
        {"conversations": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": f"Quelle est la réglementation ivoirienne en matière de {domaine} ?"},
            {"role": "assistant", "content": f"En matière de {domaine}, la {titre} constitue un texte de référence. Son article {article_num} dispose :\n\n{contenu}"}
        ]},
        {"conversations": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": f"Cite l'article {article_num} de la {titre}."},
            {"role": "assistant", "content": f"Article {article_num} — {titre} :\n\n{contenu}"}
        ]},
    ]

    if "on entend par" in contenu.lower() or chapter == "dispositions generales":
        examples.append({"conversations": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": f"Quelles définitions donne la {titre} ?"},
            {"role": "assistant", "content": f"La {titre}, article {article_num}, définit :\n\n{contenu}"}
        ]})

    return examples


def prepare_dataset(input_dir, domaines_cibles=None):
    """
    Charge tous les fichiers JSON d'un dossier (et sous-dossiers)
    """
    all_files = list(Path(input_dir).rglob("*.json"))
    print(f"{len(all_files)} fichiers JSON trouvés")

    raw_data = []
    for filepath in all_files:
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                entry = json.load(f)
            raw_data.append(entry)
        except Exception as e:
            print(f"Erreur sur {filepath.name} : {e}")

    # Filtrer par domaine si spécifié
    if domaines_cibles:
        raw_data = [
            d for d in raw_data
            if d.get("legal_field", "").lower() in [x.lower() for x in domaines_cibles]
        ]
        print(f"Domaines filtrés : {domaines_cibles} → {len(raw_data)} documents")

    all_examples = []
    for entry in raw_data:
        all_examples.extend(generate_examples(entry))

    random.shuffle(all_examples)
    split = int(len(all_examples) * 0.9)
    train_data = all_examples[:split]
    val_data   = all_examples[split:]

    print(f"Total : {len(all_examples)} | Train : {len(train_data)} | Val : {len(val_data)}")
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

In [ ]:
INPUT_DIR = "/kaggle/input/datasets/solojunior22/droit-du-travail-ivoirien/droit du travail/"
DOMAINES  = None  # None car tous les fichiers sont déjà du droit du travail

train_dataset, val_dataset = prepare_dataset(INPUT_DIR, domaines_cibles=DOMAINES)

In [ ]:
# ============================================================
# CELL 5 — Charger Gemma 4 E4B avec Unsloth
# ============================================================
model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    max_seq_length  = 2048,   
    load_in_4bit    = True,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, 
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = 16,
    lora_alpha                 = 16,
    lora_dropout               = 0,
    bias                       = "none",
    random_state               = 3407,
)

model.print_trainable_parameters()

In [ ]:
# ============================================================
# CELL 6 — Appliquer le chat template Gemma 4
# ============================================================
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix("<bos>")
        for convo in examples["conversations"]
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset   = val_dataset.map(formatting_prompts_func, batched=True)

print(train_dataset[0]["text"][:500])

In [ ]:
# ============================================================
# CELL 7 — Entraînement (Kaggle 2×T4)
# ============================================================
import os
RUN_NAME = "gemma4-droit-du-travail-ivoirien"

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = None,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,        # 2 au lieu de 1 (2×T4)
        gradient_accumulation_steps = 4,
        warmup_steps                = 10,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        logging_steps               = 10,
        eval_strategy               = "no",
        save_strategy               = "steps",
        save_steps                  = 100,      # sauvegarde fréquente
        save_total_limit            = 3,
        load_best_model_at_end      = False,
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = "/kaggle/working/checkpoints",  # Kaggle working dir
        report_to                   = "none",
        max_seq_length              = 2048,     # 2048 possible avec 2×T4
        fp16                        = True,     # T4 ne supporte pas bf16
        dataloader_num_workers      = 2,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)

trainer_stats = trainer.train(resume_from_checkpoint=False)

# Sauvegarder les poids LoRA dans /kaggle/working/
model.save_pretrained(f"/kaggle/working/{RUN_NAME}-lora")
tokenizer.save_pretrained(f"/kaggle/working/{RUN_NAME}-lora")
print(f"Modèle sauvegardé dans /kaggle/working/{RUN_NAME}-lora")

In [ ]:
# CELL 8 — Sauvegarder comme Dataset Kaggle public
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

# Option 1 — Pousser directement sur HuggingFace (recommandé)
secrets   = UserSecretsClient()
hf_token  = secrets.get_secret("HF_token")
login(token=hf_token)

model.push_to_hub("solojunior22/gemma4-droit-ivoirien-lora")
tokenizer.push_to_hub("solojunior22/gemma4-droit-ivoirien-lora")
print("Modèle sauvegardé sur HuggingFace")